Лабораторна робота №2

Частина 1




In [3]:
import pandas as pd
import urllib.request
import os
from datetime import datetime

Завдання 1
Для кожної з адміністративних одиниць України завантажити (urllib) тестові структуровані файли, що містять значення VHI-індексу. При зберіганні файлу, до його імені потрібно додати дату та час завантаження. Передбачити повторні запуски скрипту, реалізувати механізм запобігання повторного довантаження та колізії даних

In [4]:
def download_vhi(province_id, folder="vhi_data"):
    
    if not os.path.exists(folder):
        os.makedirs(folder)

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    filename = f"vhi_id_{province_id}_{timestamp}.csv"
    filepath = os.path.join(folder, filename)

    url = f"https://www.star.nesdis.noaa.gov/smcd/emb/vci/VH/get_TS_admin.php?country=UKR&provinceID={province_id}&year1=1981&year2=2024&type=Mean"

    urllib.request.urlretrieve(url, filepath)

    print(f"Downloaded: {filename}")

Завдання 2
Завантажити дані для всіх адміністративних областей України.

In [5]:
for i in range(1, 28):
    if i != 0:
        download_vhi(i)

Downloaded: vhi_id_1_20260315_213711.csv
Downloaded: vhi_id_2_20260315_213713.csv
Downloaded: vhi_id_3_20260315_213714.csv
Downloaded: vhi_id_4_20260315_213715.csv
Downloaded: vhi_id_5_20260315_213716.csv
Downloaded: vhi_id_6_20260315_213718.csv
Downloaded: vhi_id_7_20260315_213720.csv
Downloaded: vhi_id_8_20260315_213722.csv
Downloaded: vhi_id_9_20260315_213730.csv
Downloaded: vhi_id_10_20260315_213732.csv
Downloaded: vhi_id_11_20260315_213733.csv
Downloaded: vhi_id_12_20260315_213734.csv
Downloaded: vhi_id_13_20260315_213735.csv
Downloaded: vhi_id_14_20260315_213738.csv
Downloaded: vhi_id_15_20260315_213740.csv
Downloaded: vhi_id_16_20260315_213741.csv
Downloaded: vhi_id_17_20260315_213742.csv
Downloaded: vhi_id_18_20260315_213743.csv
Downloaded: vhi_id_19_20260315_213744.csv
Downloaded: vhi_id_20_20260315_213745.csv
Downloaded: vhi_id_21_20260315_213747.csv
Downloaded: vhi_id_22_20260315_213748.csv
Downloaded: vhi_id_23_20260315_213749.csv
Downloaded: vhi_id_24_20260315_213750.csv
D

Завдання 3
Зчитати завантажені текстові файли у pandas dataframe. Здійснити data cleaning: прибрати зайві стовпці, заповнити пропуски, видалити зайвий текст тощо. Додати стовпчики з назвою та індексом області

In [ ]:
def clean_vhi_file(filepath, province_id):

    data = []

    with open(filepath, "r") as f:
        lines = f.readlines()

    for line in lines:

        line = line.strip()

        #пропускаємо службові рядки
        if not line or not line[0].isdigit():
            continue

        #прибираємо кому в кінці
        line = line.rstrip(",")

        parts = [x.strip() for x in line.split(",")]

        if len(parts) == 7:
            data.append(parts)

    df = pd.DataFrame(data, columns=[
        "Year","Week","SMN","SMT","VCI","TCI","VHI"
    ])

    df["province_id"] = province_id

    df = df.apply(pd.to_numeric)

    return df

Зчитати завантажені текстові файли у pandas dataframe.

In [7]:
def load_all_data(folder="vhi_data"):

    frames = []

    for file in os.listdir(folder):

        if file.endswith(".csv"):

            province_id = int(file.split("_")[2])

            df = clean_vhi_file(os.path.join(folder,file), province_id)

            frames.append(df)

    final_df = pd.concat(frames)

    return final_df

In [8]:
df = load_all_data()
df.head()

,Year,Week,SMN,SMT,VCI,TCI,VHI,province_id
0,1982,2,0.059,255.61,16.96,85.31,51.13,10
1,1982,3,0.060,258.56,16.90,74.56,45.73,10
2,1982,4,0.060,261.17,16.45,63.52,39.98,10
3,1982,5,0.061,263.92,15.86,53.34,34.60,10
4,1982,6,0.061,266.49,14.98,47.33,31.15,10


Завдання 5

Реалізувати процедуру зміни індексів областей.

У даних NOAA області індексуються за англійською абеткою (Province 1 — Cherkasy). Необхідно змінити індекси так, щоб області індексувались за українською абеткою (1 — Вінницька область).


In [9]:
def change_province_index(df):

    mapping = {
        1:22, 2:24, 3:23, 4:25, 5:3,
        6:4, 7:8, 8:9, 9:10, 10:11,
        11:12, 12:13, 13:14, 14:15,
        15:16, 16:17, 17:18, 18:19,
        19:20, 20:21, 21:5, 22:6,
        23:7, 24:1, 25:2, 26:26, 27:27
    }

    df["province_id"] = df["province_id"].replace(mapping)

    print("Індекси областей змінено")

    return df



In [19]:
df = change_province_index(df)
df.head()

Індекси областей змінено


,Year,Week,SMN,SMT,VCI,TCI,VHI,province_id
0,1982,2,0.059,255.61,16.96,85.31,51.13,20
1,1982,3,0.060,258.56,16.90,74.56,45.73,20
2,1982,4,0.060,261.17,16.45,63.52,39.98,20
3,1982,5,0.061,263.92,15.86,53.34,34.60,20
4,1982,6,0.061,266.49,14.98,47.33,31.15,20


Завдання 6

Реалізувати вибірку: ряд значень VHI для вказаної області за певний рік

In [20]:
def get_vhi_for_year(df, province, year):

    result = df[(df["province_id"] == province) & (df["Year"] == year)]

    return result

get_vhi_for_year(df,5,2006)

,Year,Week,SMN,SMT,VCI,TCI,VHI,province_id
1247,2006,1,0.069,257.07,36.05,71.29,53.67,5
1248,2006,2,0.069,256.86,39.45,72.60,56.02,5
1249,2006,3,0.068,257.01,38.40,75.02,56.71,5
1250,2006,4,0.066,257.40,33.83,76.19,55.01,5
1251,2006,5,0.064,258.15,28.28,77.40,52.84,5
...,...,...,...,...,...,...,...,...
1294,2006,48,0.087,270.50,43.89,15.38,29.64,5
1295,2006,49,0.081,269.93,43.10,13.43,28.26,5
1296,2006,50,0.079,269.04,43.50,13.94,28.72,5
1297,2006,51,0.077,267.58,45.60,16.01,30.81,5


Завдання 7

Реалізувати вибірку: ряд значень VHI за вказаний діапазон років для вибраних областей.

In [21]:
def get_vhi_range(df, provinces, start_year, end_year):

    result = df[
        (df["province_id"].isin(provinces)) &
        (df["Year"] >= start_year) &
        (df["Year"] <= end_year)
    ]

    return result

get_vhi_range(df, [5,8], 2000, 2006)

,Year,Week,SMN,SMT,VCI,TCI,VHI,province_id
935,2000,1,0.041,264.68,13.12,36.19,24.65,5
936,2000,2,0.046,264.21,18.06,36.93,27.49,5
937,2000,3,0.054,264.51,25.15,37.56,31.36,5
938,2000,4,0.067,265.00,34.61,39.96,37.28,5
939,2000,5,0.079,265.90,38.86,42.83,40.85,5
...,...,...,...,...,...,...,...,...
1294,2006,48,0.123,278.74,56.40,2.83,29.62,8
1295,2006,49,0.118,277.68,60.59,2.10,31.35,8
1296,2006,50,0.116,276.54,65.65,1.82,33.73,8
1297,2006,51,0.119,276.23,73.29,1.31,37.30,8


Завдання 8
Знайти екстремальні значення (мінімум, максимум), а також середнє значення та медіану для вибраних областей.

In [22]:
def vhi_statistics(df, provinces):

    subset = df[df["province_id"].isin(provinces)]

    print("Мінімальне значення VHI:", subset["VHI"].min())
    print("Максимальне значення VHI:", subset["VHI"].max())
    print("Середнє значення VHI:", subset["VHI"].mean())
    print("Медіана VHI:", subset["VHI"].median())

vhi_statistics(df, [3,5,8])

Мінімальне значення VHI: -1.0
Максимальне значення VHI: 92.31
Середнє значення VHI: 45.39615659955257
Медіана VHI: 45.84
